# T2 — Semantic ICD linking (bge-m3 + reranker) trên Colab

Build embedding index cho ICD-10-CM rồi đo hit@k trên dev gold.

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU** → Save.

Chạy lần lượt các cell. Xong gửi lại khối `=== ICD-10 (CHẨN_ĐOÁN) ===` cho P2.

In [ ]:
# 1) Kiểm tra GPU (phải thấy Tesla T4). Nếu lỗi -> chưa bật GPU ở Runtime.
!nvidia-smi -L

In [ ]:
# 2) Clone repo (nhánh Duckun). Repo private -> dán GitHub token (PAT).
#    Tạo PAT: GitHub > Settings > Developer settings > Personal access tokens (quyền đọc repo).
PAT = ""  #@param {type:"string"}
BRANCH = "Duckun"  #@param {type:"string"}
REPO = "tienndat1904/viettel-medreason"  #@param {type:"string"}
import os
if not os.path.isdir('/content/viettel-medreason'):
    !git clone -b {BRANCH} https://{PAT}@github.com/{REPO}.git /content/viettel-medreason
# %cd Ở TOP-LEVEL (không trong if/else) để đổi thư mục chắc chắn
%cd /content/viettel-medreason
!git pull
!git log --oneline -1

In [8]:
# 3) Cài thư viện (Colab đã có torch). Nếu lỗi version, đổi sang: !pip -q install -U FlagEmbedding
!pip -q install FlagEmbedding==1.3.2

In [9]:
# 4) (Tuỳ chọn) Nạp lại index đã lưu ở Drive để KHỎI build lại. Bỏ qua nếu build lần đầu.
%cd /content/viettel-medreason
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
src = '/content/drive/MyDrive/viettel_kb'
for f in ('icd10cm_emb.npy', 'icd10cm_index_meta.parquet'):
    p = os.path.join(src, f)
    if os.path.exists(p):
        shutil.copy(p, 'data/kb/'); print('đã nạp', f)
    else:
        print('chưa có', f, '-> sẽ build ở cell 5')

[Errno 2] No such file or directory: '/content/viettel-medreason'
/content
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
chưa có icd10cm_emb.npy -> sẽ build ở cell 5
chưa có icd10cm_index_meta.parquet -> sẽ build ở cell 5


In [10]:
# 5) Build index (lần đầu tải bge-m3 ~2.3GB, ~vài phút trên T4).
#    Nếu đã nạp từ Drive ở cell 4 thì có thể bỏ qua cell này.
#    Hết VRAM -> thêm: --batch-size 64
%cd /content/viettel-medreason
!python src/kb/build_icd_index.py

[Errno 2] No such file or directory: '/content/viettel-medreason'
/content
python3: can't open file '/content/src/kb/build_icd_index.py': [Errno 2] No such file or directory


In [ ]:
# 6) Bật semantic + đo trên 30 file dev gold (embed query + rerank cần GPU).
%cd /content/viettel-medreason
!sed -i 's/backend: lexical/backend: semantic/' configs/config.yaml
!python src/eval/eval_linking.py

In [ ]:
# 7) (Nên) Lưu index vào Drive để phiên sau khỏi build lại (dùng ở cell 4).
%cd /content/viettel-medreason
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/viettel_kb
!cp data/kb/icd10cm_emb.npy data/kb/icd10cm_index_meta.parquet /content/drive/MyDrive/viettel_kb/
print('Đã lưu index vào Drive/MyDrive/viettel_kb')

## Gửi lại cho P2
Copy khối kết quả ở cell 6:
```
=== ICD-10 (CHẨN_ĐOÁN) ===
  hit@k ... = 0.???
  top1  ... = 0.???
```
So với bản lexical hiện tại **0.495** để quyết định tinh chỉnh `icd_top_k_retrieve` / `icd_top_k_return` / `min_confidence`.